# Uydu Görüntüsü ile Deprem Hasar Tespiti
**xView2 Veri Seti | HistGradientBoostingClassifier | Binary Damage Classification**

Pipeline:
1. GeoTIFF pre/post görüntülerden feature extraction
2. Paralel cache oluşturma
3. Model eğitimi (HGB) + threshold optimizasyonu
4. Değerlendirme (ROC, PR, Confusion Matrix, Feature Importance)
5. İnteraktif hasar haritası

## 1. Kurulum ve Konfigürasyon

In [ ]:
# !pip install rasterio scikit-learn numpy scipy pillow folium matplotlib

import os, csv, json, math, pickle, base64, io, warnings
import multiprocessing as mp
from pathlib import Path
from typing import Any

import numpy as np
import rasterio
from rasterio.transform import array_bounds
from rasterio.warp import transform_bounds
from scipy.ndimage import sobel
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, roc_auc_score, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    confusion_matrix, f1_score
)
from sklearn.inspection import permutation_importance
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import folium
from PIL import Image

warnings.filterwarnings("ignore")

# ── Yollar ──────────────────────────────────────────────────────────────────
BASE_DIR    = Path("/mnt/ortak/JupyterProject")          # proje kökü
SPLIT_DIR   = BASE_DIR / "data/processed/splits/r80_10_10_seed42"
CACHE_DIR   = BASE_DIR / "data/feature_cache"
MODEL_DIR   = BASE_DIR / "models/scalar_baseline_hgb_v3"
OUTPUT_DIR  = BASE_DIR / "sunum_gorseller"

CACHE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def fix_path(p: str) -> str:
    """Windows yolunu Linux yoluna çevir."""
    return p.replace("D:\\JupyterProject", str(BASE_DIR)).replace("\\", "/")

print("Konfigürasyon tamam.")
print(f"  Base : {BASE_DIR}")
print(f"  Cache: {CACHE_DIR}")
print(f"  Model: {MODEL_DIR}")

## 2. Feature Extraction Fonksiyonları

In [ ]:
def _safe_float(v, default=0.0):
    try: return float(v) if v else default
    except: return default

def _signed_log1p(x):
    return math.copysign(math.log1p(abs(x)), x)

def _skewness_and_kurtosis(arr):
    flat = arr.astype(np.float64).ravel()
    if flat.size == 0: return 0.0, 0.0
    std = float(np.std(flat))
    if std <= 1e-12: return 0.0, 0.0
    z = (flat - flat.mean()) / std
    sk = float(np.mean(z**3)); ku = float(np.mean(z**4) - 3)
    return (sk if math.isfinite(sk) else 0.0), (ku if math.isfinite(ku) else 0.0)

def _ssim(a, b):
    a, b = a.astype(np.float64), b.astype(np.float64)
    mu_a, mu_b = a.mean(), b.mean()
    var_a = ((a-mu_a)**2).mean(); var_b = ((b-mu_b)**2).mean()
    cov = ((a-mu_a)*(b-mu_b)).mean()
    c1, c2 = 6.5025, 58.5225
    num = (2*mu_a*mu_b+c1)*(2*cov+c2)
    den = (mu_a**2+mu_b**2+c1)*(var_a+var_b+c2)
    return float(np.clip(num/den, -1, 1)) if abs(den) > 1e-12 else 1.0

def _edge_change(pre, post):
    pre_f, post_f = pre.astype(np.float64), post.astype(np.float64)
    pre_e  = np.hypot(sobel(pre_f,0),  sobel(pre_f,1))
    post_e = np.hypot(sobel(post_f,0), sobel(post_f,1))
    return float(np.abs(post_e-pre_e).mean() / (pre_e.mean()+1e-6))

def _band_features(pre, post):
    """18 feature per band."""
    diff = np.abs(post - pre)
    flat = diff.ravel()
    sk, ku = _skewness_and_kurtosis(diff)
    p10,p25,p75,p90 = (float(v) for v in np.percentile(flat,[10,25,75,90]))
    h,w = diff.shape; mh,mw = h//2,w//2
    return [
        float(np.mean(pre)), float(np.mean(post)), float(np.mean(diff)),
        float(np.std(pre)),  float(np.std(post)),  float(np.std(diff)),
        sk, ku, _ssim(pre,post), _edge_change(pre,post),
        p10, p25, p75, p90,
        float(diff[:mh,:mw].mean()), float(diff[:mh,mw:].mean()),
        float(diff[mh:,:mw].mean()), float(diff[mh:,mw:].mean()),
    ]

def compute_change_features(pre_path, post_path, max_bands=3):
    """54 image feature (3 band × 18)."""
    with rasterio.open(pre_path) as ds:  pre  = ds.read(out_dtype="float32")
    with rasterio.open(post_path) as ds: post = ds.read(out_dtype="float32")
    n = min(pre.shape[0], post.shape[0], max_bands)
    feats = []
    for i in range(n): feats.extend(_band_features(pre[i], post[i]))
    feats.extend([0.0]*((max_bands-n)*18))
    return feats

def build_scalar_features(pga=0.0, magnitude=0.0, depth_km=0.0,
                           acq_delta=0.0, building_count=0.0, normalize_pga=True):
    """5 scalar feature."""
    return [_signed_log1p(pga) if normalize_pga else pga,
            magnitude, depth_km, acq_delta, building_count]

FEATURE_NAMES = (
    [f"B{b+1}_{n}" for b in range(3)
     for n in ["pre_mean","post_mean","diff_mean","pre_std","post_std","diff_std",
               "diff_skew","diff_kurt","ssim","edge_change",
               "p10","p25","p75","p90","q_TL","q_TR","q_BL","q_BR"]]
    + ["pga","magnitude","depth_km","acq_delta","building_count"]
)
print(f"Feature sayısı: {len(FEATURE_NAMES)}")

## 3. Feature Cache Oluşturma
> Cache zaten varsa bu adım atlanır (~20 dakika sürüyor, 8 worker ile)

In [ ]:
def _process_row(row):
    pre  = fix_path(row["pre_image_path"])
    post = fix_path(row["post_image_path"])
    if not Path(pre).exists(): return None, None
    try:
        img = compute_change_features(pre, post)
        scl = build_scalar_features(
            pga=_safe_float(row.get("pga_value")),
            magnitude=_safe_float(row.get("magnitude")),
            depth_km=_safe_float(row.get("depth_km")),
            acq_delta=_safe_float(row.get("acquisition_delta_days")),
            building_count=_safe_float(row.get("building_count")),
        )
        x = img + scl
        if any(not math.isfinite(v) for v in x): return None, None
        return x, int(row["binary_damage"])
    except: return None, None

def build_cache(split_name, n_workers=8):
    out_x = CACHE_DIR / f"{split_name}_X.npy"
    out_y = CACHE_DIR / f"{split_name}_y.npy"
    if out_x.exists():
        print(f"{split_name}: cache mevcut ({np.load(out_x).shape[0]} satır)")
        return

    csv_path = SPLIT_DIR / f"{split_name}_split.csv"
    with open(csv_path, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    print(f"{split_name}: {len(rows)} satır, {n_workers} worker...")

    mp.set_start_method("fork", force=True)
    with mp.Pool(n_workers) as pool:
        results = pool.map(_process_row, rows)

    X = [r[0] for r in results if r[0] is not None]
    y = [r[1] for r in results if r[1] is not None]
    np.save(out_x, np.array(X, dtype=np.float32))
    np.save(out_y, np.array(y, dtype=np.int64))
    print(f"  {len(y)} örnek, {len(X[0])} feature kaydedildi.")

# Cache oluştur (yoksa)
if __name__ == "__main__":
    for split in ["train", "val", "test"]:
        build_cache(split)

## 4. Cache'den Veri Yükleme ve Stratified Split

In [ ]:
# Tüm cache'i birleştir
X_all = np.concatenate([np.load(CACHE_DIR/f"{s}_X.npy") for s in ["train","val","test"]])
y_all = np.concatenate([np.load(CACHE_DIR/f"{s}_y.npy") for s in ["train","val","test"]])
print(f"Toplam: {len(y_all)} örnek, {X_all.shape[1]} feature")
print(f"Hasar oranı: {y_all.mean():.1%}")

# Stratified 80/10/10
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X_all, y_all, test_size=0.2, stratify=y_all, random_state=42)
X_va, X_te, y_va, y_te   = train_test_split(X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=42)

for name, y in [("Train",y_tr),("Val",y_va),("Test",y_te)]:
    print(f"  {name}: {len(y)} örnek, hasar={y.mean():.1%}")

## 5. Model Eğitimi

In [ ]:
# Eğitilmiş modeli yükle (yoksa eğit)
MODEL_PKL = MODEL_DIR / "model.pkl"
INFERENCE_CFG = MODEL_DIR / "inference_config.json"

if MODEL_PKL.exists():
    with MODEL_PKL.open("rb") as f: model = pickle.load(f)
    best_t = json.loads(INFERENCE_CFG.read_text())["threshold"]
    print(f"Model yüklendi: {MODEL_PKL}")
    print(f"Threshold: {best_t}")
else:
    # hgb_v3 hiperparametreleri (grid search ile seçildi)
    model = HistGradientBoostingClassifier(
        max_iter=300,
        learning_rate=0.03,
        max_depth=6,
        min_samples_leaf=150,
        l2_regularization=1.0,
        random_state=42,
    )
    model.fit(X_tr, y_tr)
    print("Eğitim tamamlandı.")

    # Val üzerinde threshold optimizasyonu
    probs_va = model.predict_proba(X_va)[:,1]
    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.25, 0.65, 0.01):
        f1 = f1_score(y_va, (probs_va >= t).astype(int))
        if f1 > best_f1: best_f1, best_t = f1, t
    print(f"Optimum threshold: {best_t:.2f}  (val F1={best_f1:.3f})")

    # Modeli kaydet
    with MODEL_PKL.open("wb") as f: pickle.dump(model, f)
    INFERENCE_CFG.write_text(json.dumps({
        "threshold": round(float(best_t), 2),
        "max_bands": 3, "image_only": False, "normalize_pga": True
    }))
    print(f"Model kaydedildi: {MODEL_DIR}")

## 6. Test Seti Değerlendirmesi

In [ ]:
probs_te = model.predict_proba(X_te)[:,1]
preds_te = (probs_te >= best_t).astype(int)

print(classification_report(y_te, preds_te, target_names=["Sağlam","Hasar"]))
print(f"ROC-AUC: {roc_auc_score(y_te, probs_te):.4f}")

## 7. Değerlendirme Grafikleri

In [ ]:
BG="#0f0f1a"; CARD="#1a1a2e"; ACC="#e94560"; GRN="#0f9b8e"; YEL="#f5a623"; BLU="#4a90d9"

def savefig(name):
    path = OUTPUT_DIR/name
    plt.savefig(path, dpi=130, bbox_inches="tight", facecolor=plt.gcf().get_facecolor())
    plt.close()
    print(f"  {name}")

# Confusion Matrix
cm = confusion_matrix(y_te, preds_te)
fig, ax = plt.subplots(figsize=(5,4)); fig.patch.set_facecolor(BG); ax.set_facecolor(CARD)
ax.imshow(cm, cmap="RdYlGn")
for i in range(2):
    for j in range(2):
        ax.text(j,i,str(cm[i,j]),ha="center",va="center",fontsize=22,fontweight="bold",color="white")
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(["Tahmin: Sağlam","Tahmin: Hasar"],color="white",fontsize=11)
ax.set_yticklabels(["Gerçek: Sağlam","Gerçek: Hasar"],color="white",fontsize=11)
tp_v,fn_v,fp_v,tn_v = cm[1,1],cm[1,0],cm[0,1],cm[0,0]
pr=tp_v/(tp_v+fp_v); rc=tp_v/(tp_v+fn_v); f1=2*pr*rc/(pr+rc)
ax.set_title(f"Confusion Matrix   F1={f1:.2f}  Prec={pr:.2f}  Rec={rc:.2f}",color="white",fontsize=11,pad=10)
for sp in ax.spines.values(): sp.set_visible(False)
plt.tight_layout(); savefig("1_confusion_matrix.png")

# ROC
fpr,tpr,_ = roc_curve(y_te,probs_te); roc_auc=auc(fpr,tpr)
fig,ax=plt.subplots(figsize=(5,4)); fig.patch.set_facecolor(BG); ax.set_facecolor(CARD)
ax.plot(fpr,tpr,color=ACC,lw=2.5,label=f"AUC={roc_auc:.3f}")
ax.plot([0,1],[0,1],color="#555",lw=1,linestyle="--")
ax.fill_between(fpr,tpr,alpha=0.15,color=ACC)
ax.set_xlabel("False Positive Rate",color="white"); ax.set_ylabel("True Positive Rate",color="white")
ax.set_title("ROC Eğrisi",color="white",fontsize=13); ax.legend(facecolor=CARD,labelcolor="white")
ax.tick_params(colors="white"); ax.spines[:].set_color("#333")
plt.tight_layout(); savefig("2_roc_curve.png")

# Precision-Recall
prec_c,rec_c,_ = precision_recall_curve(y_te,probs_te); ap=average_precision_score(y_te,probs_te)
fig,ax=plt.subplots(figsize=(5,4)); fig.patch.set_facecolor(BG); ax.set_facecolor(CARD)
ax.plot(rec_c,prec_c,color=GRN,lw=2.5,label=f"AP={ap:.3f}")
ax.fill_between(rec_c,prec_c,alpha=0.15,color=GRN)
ax.axvline(rc,color=YEL,lw=1.5,linestyle="--",label=f"Threshold={best_t:.2f}")
ax.set_xlabel("Recall",color="white"); ax.set_ylabel("Precision",color="white")
ax.set_title("Precision-Recall Eğrisi",color="white",fontsize=13); ax.legend(facecolor=CARD,labelcolor="white")
ax.tick_params(colors="white"); ax.spines[:].set_color("#333")
plt.tight_layout(); savefig("3_pr_curve.png")

# Feature Importance (permutation — ~1 dakika)
print("Feature importance hesaplanıyor...")
r = permutation_importance(model, X_te, y_te, n_repeats=5, random_state=42, n_jobs=-1)
imp = r.importances_mean
top_idx = np.argsort(imp)[-20:]
fig,ax=plt.subplots(figsize=(7,6)); fig.patch.set_facecolor(BG); ax.set_facecolor(CARD)
colors=[ACC if any(k in FEATURE_NAMES[i] for k in ["building","pga","magnitude"])
        else GRN if any(k in FEATURE_NAMES[i] for k in ["ssim","edge"]) else BLU for i in top_idx]
ax.barh([FEATURE_NAMES[i] for i in top_idx],imp[top_idx],color=colors,alpha=0.9)
patches=[mpatches.Patch(color=ACC,label="Skalar"),mpatches.Patch(color=GRN,label="SSIM/Edge"),
         mpatches.Patch(color=BLU,label="İstatistiksel")]
ax.legend(handles=patches,facecolor=CARD,labelcolor="white",fontsize=9)
ax.set_xlabel("Permutation Önem",color="white"); ax.set_title("En Önemli 20 Özellik",color="white",fontsize=13)
ax.tick_params(colors="white",labelsize=9); ax.spines[:].set_color("#333")
plt.tight_layout(); savefig("5_feature_importance.png")

# Olasılık Dağılımı
all_probs_plot = model.predict_proba(X_all)[:,1]
fig,ax=plt.subplots(figsize=(6,4)); fig.patch.set_facecolor(BG); ax.set_facecolor(CARD)
ax.hist(all_probs_plot[y_all==0],bins=50,alpha=0.75,color=GRN,label="Sağlam",density=True)
ax.hist(all_probs_plot[y_all==1],bins=50,alpha=0.75,color=ACC,label="Hasar", density=True)
ax.axvline(best_t,color=YEL,lw=2,linestyle="--",label=f"Eşik={best_t}")
ax.set_xlabel("Tahmin Olasılığı",color="white"); ax.set_ylabel("Yoğunluk",color="white")
ax.set_title("Hasar Olasılığı Dağılımı",color="white",fontsize=13)
ax.legend(facecolor=CARD,labelcolor="white"); ax.tick_params(colors="white"); ax.spines[:].set_color("#333")
plt.tight_layout(); savefig("6_prob_dagilimi.png")
print("Grafikler tamamlandı.")

## 8. İnteraktif Hasar Haritası
> Tile'a tıklayınca **Öncesi / Sonrası** görüntüleri açılır

In [ ]:
def read_rgb_b64(path):
    """GeoTIFF → base64 PNG (256x256)."""
    with rasterio.open(path) as ds:
        arr = ds.read(out_dtype="uint8")
    rgb = np.moveaxis(arr[:3],0,-1) if arr.shape[0]>=3 else np.stack([arr[0]]*3,axis=-1)
    img = Image.fromarray(rgb).resize((256,256))
    buf = io.BytesIO(); img.save(buf,format="PNG")
    return base64.b64encode(buf.getvalue()).decode()

def build_demo_map(csv_path, event_id, max_tiles=200):
    tiles = []
    with open(csv_path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            if row["event_id"] != event_id: continue
            pre = fix_path(row["pre_image_path"])
            if not Path(pre).exists(): continue
            tiles.append({"pre":pre,"post":fix_path(row["post_image_path"]),
                          "true":int(row["binary_damage"]),
                          "bc":_safe_float(row.get("building_count"))})
            if len(tiles) >= max_tiles: break

    results = []
    for t in tiles:
        try:
            img = compute_change_features(t["pre"], t["post"])
            scl = build_scalar_features(building_count=t["bc"])
            prob = float(model.predict_proba(np.array([img+scl],dtype=np.float32))[0,1])
            with rasterio.open(t["pre"]) as ds:
                b = array_bounds(ds.height,ds.width,ds.transform)
                w84 = transform_bounds(ds.crs,"EPSG:4326",*b)
            results.append({"bounds":[[w84[1],w84[0]],[w84[3],w84[2]]],
                            "prob":prob,"pred":int(prob>=best_t),"true":t["true"],
                            "pre":t["pre"],"post":t["post"]})
        except: pass

    lats=[(r["bounds"][0][0]+r["bounds"][1][0])/2 for r in results]
    lons=[(r["bounds"][0][1]+r["bounds"][1][1])/2 for r in results]
    m = folium.Map(location=[np.mean(lats),np.mean(lons)],zoom_start=13,
                   tiles="https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}",attr="Google")

    for r in results:
        if   r["pred"]==1 and r["true"]==1: c,lbl="#ff0000","Doğru Hasar (TP)"
        elif r["pred"]==1 and r["true"]==0: c,lbl="#ff9900","Yanlış Alarm (FP)"
        elif r["pred"]==0 and r["true"]==1: c,lbl="#3333ff","Kaçırılan (FN)"
        else:                               c,lbl="#00cc00","Doğru Sağlam (TN)"
        pre_b64  = read_rgb_b64(r["pre"])
        post_b64 = read_rgb_b64(r["post"])
        popup_html = f"""
        <div style="font-family:sans-serif;width:540px">
          <div style="font-size:15px;font-weight:bold;color:{c};margin-bottom:8px">
            {lbl} &nbsp; <span style="color:#333">Olasılık: {r['prob']:.0%}</span>
          </div>
          <div style="display:flex;gap:8px">
            <div style="text-align:center">
              <div style="font-size:11px;color:#666;margin-bottom:4px">ÖNCE</div>
              <img src="data:image/png;base64,{pre_b64}" width="256" height="256"
                   style="border:2px solid #ccc;border-radius:4px"/>
            </div>
            <div style="text-align:center">
              <div style="font-size:11px;color:#666;margin-bottom:4px">SONRA</div>
              <img src="data:image/png;base64,{post_b64}" width="256" height="256"
                   style="border:2px solid {c};border-radius:4px"/>
            </div>
          </div>
          <div style="font-size:11px;color:#888;margin-top:6px">
            Gerçek: {'Hasar' if r['true'] else 'Sağlam'} | Tahmin: {'Hasar' if r['pred'] else 'Sağlam'}
          </div>
        </div>"""
        folium.Rectangle(bounds=r["bounds"],color=c,weight=1,fill=True,
                         fill_color=c,fill_opacity=0.5 if r["pred"] else 0.2,
                         popup=folium.Popup(popup_html,max_width=580),
                         tooltip=f"{lbl} | {r['prob']:.0%}").add_to(m)

    legend = """<div style="position:fixed;bottom:30px;left:30px;z-index:1000;background:white;
    padding:12px;border-radius:8px;font-size:13px;box-shadow:2px 2px 8px rgba(0,0,0,0.3)">
    <b>Model Tahminleri</b><br><i style="font-size:11px;color:#666">Tile'a tıkla → öncesi/sonrası</i><br><br>
    <span style="color:#ff0000">■</span> Doğru Hasar (TP)<br>
    <span style="color:#ff9900">■</span> Yanlış Alarm (FP)<br>
    <span style="color:#3333ff">■</span> Kaçırılan (FN)<br>
    <span style="color:#00cc00">■</span> Doğru Sağlam (TN)</div>"""
    m.get_root().html.add_child(folium.Element(legend))
    return m, results

m, results = build_demo_map(SPLIT_DIR/"test_split.csv", "socal-fire", max_tiles=200)
tp=sum(1 for r in results if r["pred"]==1 and r["true"]==1)
fp=sum(1 for r in results if r["pred"]==1 and r["true"]==0)
tn=sum(1 for r in results if r["pred"]==0 and r["true"]==0)
fn=sum(1 for r in results if r["pred"]==0 and r["true"]==1)
print(f"TP={tp} FP={fp} TN={tn} FN={fn}")
out_html = BASE_DIR/"socal_demo.html"
m.save(str(out_html))
print(f"Harita kaydedildi: {out_html}")
m   # Jupyter'da göster

## 9. Yeni Görüntü Üzerinde Inference

In [ ]:
def predict(pre_image_path: str, post_image_path: str,
            pga: float = 0.0, magnitude: float = 0.0,
            depth_km: float = 0.0, building_count: float = 0.0) -> dict:
    """İki GeoTIFF verilince hasar tahmini döner."""
    img = compute_change_features(pre_image_path, post_image_path)
    scl = build_scalar_features(pga=pga, magnitude=magnitude,
                                 depth_km=depth_km, building_count=building_count)
    prob = float(model.predict_proba(np.array([img+scl],dtype=np.float32))[0,1])
    return {
        "probability":  round(prob, 4),
        "binary_damage": int(prob >= best_t),
        "threshold":    best_t,
        "label":        "HASAR" if prob >= best_t else "SAĞLAM"
    }

# Örnek kullanım:
# result = predict("pre.tif", "post.tif", pga=0.8, building_count=15)
# print(result)
print("Fonksiyon hazır. predict(pre.tif, post.tif) ile çağır.")